# Clase 4 · El taller, por enumeración

Este cuaderno hace tres cosas:

1. Una función que recibe un modelo entero —`c`, `A`, `b` y las cotas— y lo
   resuelve **enumerando**, exactamente como el pseudocódigo de la página 3.
2. Resuelve el problema del taller y **las tres variantes** de la página 2.
3. Mide cuánto cuesta, y grafica cómo crece con el tamaño del problema.

Corre las celdas en orden. Cada una tarda menos de un segundo, salvo la de la
gráfica.

## Convenciones, que son la fuente número uno de errores al pasar del papel aquí

| En las notas | Aquí |
|---|---|
| Las desigualdades van `A x <= b` | igual: `A @ x <= b` |
| Los problemas son de **máximo** | `enumerar` maximiza directamente |
| `scipy` **siempre minimiza** | al comparar con `milp` se pasa `-c`, y al óptimo se le cambia el signo |
| El dominio es entero | las cotas `l` y `u` son enteras y `range` las recorre |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from time import perf_counter
from collections import namedtuple

Modelo = namedtuple("Modelo", "c A b l u")
Resultado = namedtuple("Resultado", "x z candidatos factibles orden segundos")

## 1 · La caja, deducida del modelo

La caja no es un dato: sale de las restricciones. Si todos los coeficientes de
un renglón son $\ge 0$, entonces $a_{ij}x_j \le b_i$, y de ahí
$x_j \le \lfloor b_i / a_{ij} \rfloor$.

**Solo valen los renglones con todos los coeficientes no negativos.** Un renglón
con un coeficiente negativo —como los enlaces de las variantes— no acota nada por
sí solo, así que esas cotas hay que declararlas (una binaria es `u = 1`).

In [ ]:
def caja(A, b):
    """Cota superior de cada variable, deducida de las restricciones.

    Devuelve None en la posicion j si ningun renglon util acota a x_j: ahi la
    caja es infinita y enumerar no se puede usar.
    """
    A, b = np.asarray(A), np.asarray(b)
    topes = []
    for j in range(A.shape[1]):
        candidatas = [b[i] // A[i, j]
                      for i in range(A.shape[0])
                      if A[i, j] > 0 and np.all(A[i] >= 0)]
        topes.append(int(min(candidatas)) if candidatas else None)
    return topes


def modelo(c, A, b, u=None, l=None):
    """Arma un Modelo; si no le das u, la deduce de las restricciones."""
    u = caja(A, b) if u is None else list(u)
    if any(t is None for t in u):
        raise ValueError(f"la caja es infinita en {[j for j, t in enumerate(u) if t is None]}: "
                         "declara una cota superior o enumerar no aplica")
    l = [0] * len(u) if l is None else list(l)
    return Modelo(np.asarray(c), np.asarray(A), np.asarray(b), l, u)

## 2 · Enumerar

Es el pseudocódigo de la página 3, línea por línea. Los comentarios `L1`…`L7`
son las mismas etiquetas del diagrama de flujo.

In [ ]:
def enumerar(m, traza=False):
    reloj = perf_counter()
    mejor, x_mejor, orden = -np.inf, None, None          # L1
    candidatos = factibles = 0
    filas = []
    for punto in product(*[range(li, ui + 1) for li, ui in zip(m.l, m.u)]):   # L2, L3
        candidatos += 1
        x = np.array(punto)
        cabe = bool(np.all(m.A @ x <= m.b))              # L4 - filtra
        z = int(m.c @ x)                                 # L5 - compara
        if cabe:
            factibles += 1
            if z > mejor:                                # L6
                mejor, x_mejor, orden = z, punto, candidatos   # L7 - guarda
        if traza:
            filas.append((candidatos, punto, cabe, z, mejor))
    r = Resultado(x_mejor, mejor, candidatos, factibles, orden, perf_counter() - reloj)
    return (r, filas) if traza else r


def informe(nombre, r):
    print(f"{nombre}")
    print(f"  optimo      x* = {r.x},  z* = {r.z}")
    print(f"  candidatos  {r.candidatos}   factibles {r.factibles}")
    print(f"  aparecio    en el candidato {r.orden} de {r.candidatos}")
    print(f"  tiempo      {r.segundos*1000:.1f} ms")

## 3 · El taller

$$\max\; 5x_1 + 4x_2 \quad\text{s.a.}\quad 6x_1 + 4x_2 \le 24,\quad x_1 + 2x_2 \le 6,\quad x \in \mathbb{Z}^2_{\ge 0}$$

In [ ]:
taller = modelo(c=[5, 4], A=[[6, 4], [1, 2]], b=[24, 6])
print("caja deducida:", taller.u, "->", np.prod([u - l + 1 for l, u in zip(taller.l, taller.u)]), "candidatos")

r, filas = enumerar(taller, traza=True)
informe("taller", r)

### La rejilla de la página, impresa

Cada celda es un plan. Entre paréntesis, el orden en que lo mira el ciclo.

In [ ]:
def rejilla(m, filas):
    valor = {(p[0], p[1]): (n, ok, z) for n, p, ok, z, _ in filas}
    print("        " + "".join(f"x1={j:<8}" for j in range(m.u[0] + 1)))
    for i in range(m.u[1], m.l[1] - 1, -1):
        celdas = []
        for j in range(m.u[0] + 1):
            n, ok, z = valor[(j, i)]
            celdas.append(f"{z:>3} ({n:>2})  " if ok else f"  x ({n:>2})  ")
        print(f"x2={i}    " + "".join(celdas))

rejilla(taller, filas)

## 4 · Las tres variantes de la página 2

Las tres agregan algo que no es un renglón más. Fíjate en que **el modelo es lo
único que cambia**: `enumerar` no se entera.

In [ ]:
# Variante 1 - encender la linea cuesta 3 horas.  y es indicadora, enlace x1 <= 4y
v1 = modelo(c=[5, 4, 0],
            A=[[6, 4, 0],      # aleacion
               [1, 2, 3],      # calibracion, con las 3 horas de encendido
               [1, 0, -4]],    # enlace: x1 - 4y <= 0
            b=[24, 6, 0], u=[4, 3, 1])

# Variante 2 - o ninguno, o al menos tres.  3y <= x1 <= 4y
v2 = modelo(c=[5, 4, 0],
            A=[[6, 4, 0],
               [1, 2, 0],
               [1, 0, -4],     # x1 <= 4y
               [-1, 0, 3]],    # 3y <= x1
            b=[24, 6, 0, 0], u=[4, 3, 1])

# Variante 3 - la antena se comparte: x1 = u + v, los dos primeros a 5 MB y el resto a 3
v3 = modelo(c=[5, 3, 4],
            A=[[6, 6, 4],
               [1, 1, 2]],
            b=[24, 6], u=[2, 2, 3])

for nombre, m in [("v1 encender cuesta", v1), ("v2 o ninguno o tres", v2), ("v3 antena compartida", v3)]:
    informe(nombre, enumerar(m))
    print()

Léelas contra la respuesta de la página: el taller da 20, encender cuesta baja a
15, el lote no estorba y sigue en 20, y la antena compartida cae a 18.

En `v1` y `v2` la tercera coordenada es la indicadora `y`, no una cantidad.
En `v3` las dos primeras son los tramos `u` y `v`: el número de rovers es su suma.

## 5 · Contra el solucionador

`scipy.optimize.milp` no enumera: es branch and cut. Si coincide, la enumeración
no mintió.

In [ ]:
from scipy.optimize import milp, LinearConstraint, Bounds

def con_milp(m):
    r = milp(c=-np.asarray(m.c, dtype=float),          # scipy minimiza: se le pasa -c
             constraints=LinearConstraint(m.A, -np.inf, m.b),
             integrality=np.ones(len(m.c)),
             bounds=Bounds(m.l, m.u))
    return np.round(r.x).astype(int), int(round(-r.fun))  # y al optimo se le cambia el signo

for nombre, m in [("taller", taller), ("v1", v1), ("v2", v2), ("v3", v3)]:
    x_m, z_m = con_milp(m)
    r = enumerar(m)
    print(f"{nombre:8} enumerar z*={r.z:3}   milp z*={z_m:3}   coinciden: {r.z == z_m}")

## 6 · Qué cuesta, y cómo crece

Hay cuatro cosas que pueden crecer —$n$, $m$, y las cotas $l$ y $u$— así que se
mueve **una a la vez y las demás quedan fijas**. Dos experimentos:

| Experimento | Qué mueve | Qué fija | Qué esperamos |
|---|---|---|---|
| A | El número de variables $n$ | $m=3$, todas 0/1 | $2^n$: **exponencial** |
| B | El número de restricciones $m$ | $n=12$, todas 0/1 | $O(mn)$ por candidato: **proporcional** |

Las instancias son aleatorias con semilla fija, así que el resultado se repite.

In [ ]:
rng = np.random.default_rng(7)

def instancia(n, m):
    """Mochila binaria aleatoria con n variables y m restricciones."""
    A = rng.integers(1, 10, size=(m, n))
    b = A.sum(axis=1) // 2
    c = rng.integers(1, 10, size=n)
    return modelo(c, A, b, u=[1] * n)

def cronometrar(m, repeticiones=3):
    """El minimo de varias corridas: el ruido solo puede sumar tiempo."""
    return min(enumerar(m).segundos for _ in range(repeticiones))

# A - mover n, con m fijo
ns = list(range(4, 17))
t_n = [enumerar(instancia(n, 3)).segundos for n in ns]

# B - mover m, con n fijo. Hacen falta CIENTOS de restricciones para que se note.
ms = [1, 50, 100, 200, 300, 400]
t_m = [cronometrar(instancia(10, m)) * 1000 for m in ms]

fig, (a, b_) = plt.subplots(1, 2, figsize=(9.5, 3.4))

a.semilogy(ns, t_n, "o-", label="medido")
a.semilogy(ns, [t_n[-1] * 2.0 ** (n - ns[-1]) for n in ns], "--", color="gray",
           label="proporcional a $2^n$")
a.set_xlabel("variables $n$   ($m=3$)"); a.set_ylabel("segundos (log)")
a.set_title("A · mover $n$"); a.legend(); a.grid(alpha=.3)

pend, piso = np.polyfit(ms, t_m, 1)
b_.plot(ms, t_m, "o-", label="medido")
b_.plot(ms, [piso + pend * m for m in ms], "--", color="gray", label="recta ajustada")
b_.axhline(piso, color="gray", lw=.8, ls=":")
b_.annotate(f"piso: {piso:.1f} ms de costo fijo", (ms[1], piso), textcoords="offset points",
            xytext=(0, 8), fontsize=8, color="gray")
b_.set_xlabel("restricciones $m$   ($n=10$)"); b_.set_ylabel("milisegundos")
b_.set_title("B · mover $m$"); b_.legend(fontsize=8); b_.grid(alpha=.3)

fig.tight_layout(); plt.show()

print(f"A · n de {ns[0]} a {ns[-1]}: el tiempo se multiplico por {t_n[-1]/t_n[0]:,.0f}")
print(f"B · m de {ms[0]} a {ms[-1]} (400 veces mas restricciones): "
      f"el tiempo se multiplico por {t_m[-1]/t_m[0]:.1f}")

**Lee las dos escalas, que no son la misma.**

La izquierda es **logarítmica** y aun así la curva sube como una recta: eso *es*
crecimiento exponencial. Doce variables más multiplican el tiempo por miles.

La derecha es lineal, y dice dos cosas. Sube **recta**, como manda el $O(mn)$ —
pero **no arranca en cero**: ese piso es el costo fijo de cada candidato, que en
Python (crear el punto, llamar a numpy) pesa más que las $mn$ multiplicaciones
hasta bien entradas las cien restricciones. La constante es grande; la forma, no.

Y ésa es la comparación que hay que guardarse: multiplicar las restricciones por
400 apenas duplica el tiempo. **$m$ sube por una escalera; $n$ sube por un
precipicio.**

Cuando veas el método que no mira todos los candidatos, la comparación se hace
contra esta misma gráfica.

## 7 · Tu turno

> **Bitácora — día 231.** El invernadero reparte **12 m² de sustrato** y
> **10 litros de nutriente** al día entre charolas de tomate y de lechuga. Una
> charola de tomate ocupa 3 m², bebe 2 L y da 7 raciones; una de lechuga ocupa
> 2 m², bebe 3 L y da 5 raciones. Las charolas no se parten por la mitad, y en
> el almacén **solo quedan 3 charolas de tomate**.

Escribe el modelo abajo —`c`, `A`, `b`— y corre la celda. El valor esperado está
codificado a propósito, para que no lo leas de un vistazo.

In [ ]:
import hashlib

# Escribe aqui tu modelo. Un renglon de A por restriccion, en el mismo orden que b.
c = [...]          # raciones por charola: tomate, lechuga
A = [[...], [...]] # sustrato, nutriente, y lo que falte
b = [...]          # lo disponible de cada uno

sin_llenar = (Ellipsis in c or Ellipsis in b
              or any(Ellipsis in fila for fila in A))

if sin_llenar:
    print("escribe tu modelo arriba y vuelve a correr la celda")
else:
    mio = modelo(c, A, b)
    r = enumerar(mio)
    informe("tu modelo", r)
    ok = hashlib.sha256(str(r.z).encode()).hexdigest()[:8] == "5f9c4ab0"
    print()
    print("correcto" if ok else
          "el optimo no es el esperado: revisa que esten las TRES restricciones")